In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, cellprofiler_results, require)
from utils.panels import save_panel

import os
import pandas as pd 

os.getcwd()


In [ ]:

sourceDir = str(cellprofiler_results("exp1_main"))
rootDir = str(data_dir("exp1_main")) + "/"


### Define Functions


In [ ]:
metaEx = pd.read_csv(f'{rootDir}spher_colo52-metadata.csv')
these_cols = ['barcode','well_id', 'cmpdname', 'solvent', 'cmpd_conc', 'target', 'pathway', 'pubchemID', 'inkey', 'cell_line', 'article_id']
metaEx = metaEx[these_cols]

# Rename a few columns
metaEx = metaEx.rename(columns={'cmpd_conc': 'cmpd_conc_um', 'article_id': 'selleckchem_id'})

metaEx.head()

In [ ]:
Template = pd.read_csv(f'spher-colo52.tsv', sep='\t')

In [ ]:
Template.head()

# Add a new column wiht the barcode (starts with PB0001..)
Template['barcode'] = Template['Files'].str.slice(13, 21)

# # Regex to extract from the filename
Template[['well_id', 'z_plane', 'channel']] = Template['Files'].str.extract(r'Well-([A-Z0-9]{3})-z([0-9]{1,2})-(.*).ome.tiff')

Template.head()



In [ ]:
# Merge the two dataframes on barcode and well_id
FileList = pd.merge(Template, metaEx, on=['barcode', 'well_id'], how='left')

FileList.head()



In [ ]:
# The detection example dataset (Nikon spheroids-revision acquisition)
# is part of the same submission, so it goes into the same file list. It is a
# flat folder with no plate barcode, HOECHST + SYTO only, and no segmentation
# masks — so it contributes images here but nothing to the annotations file.
Detection = pd.read_csv('detection_example_dataset.tsv', sep='\t')

# Same filename regex as the plates, minus the barcode (there is none here)
Detection[['well_id', 'z_plane', 'channel']] = Detection['Files'].str.extract(
    r'Well-([A-Z0-9]{3})-z([0-9]{1,2})-(.*).ome.tiff')
Detection['cell_line'] = 'HCT116'  # same spelling as the plate metadata

assert Detection['well_id'].notna().all(), 'some filenames did not parse'
print(len(Detection), 'files |',
      Detection['well_id'].nunique(), 'wells |',
      Detection['z_plane'].astype(int).max() + 1, 'z planes |',
      sorted(Detection['channel'].unique()))

Detection.head()


### Still to deposit

The EdU per-object tables are now in the deposit and reach the file list through the
manifest, like every other processed table — nothing extra to add for them.

One tier is still outside it:

- **the 2D monolayer profiles** (31 MB) — `selected_data_{HCT116,HT29}.csv`. No panel
  reads them: they only regenerated `selected_data_2D_*` and `grit_data_2D_*`, which are
  already deposited, so both they and the notebook that read them have been removed from
  this tree. Depositing them closes the 2D branch of the provenance chain.

Add it to `supplementary` below once uploaded, then re-run this notebook so
`spher_colo52-FileList.tsv` names it.

`input/expert_annotation/` (157 MB) is bound for the archive too, but as an annotation
tier rather than a processed table; it is not carried by the manifest.

In [ ]:
# BIA takes a single file list per submission, so everything is concatenated
# into one table: the plate images, then the detection example images, then the
# supplementary (non-image) files uploaded alongside them — CellProfiler
# runs/pipeline, Nikon acquisition macros, the per-plate feature tables, and the
# processed profile tables the figures are built from.
# Columns that do not apply to a block are left empty.
barcodes = sorted(FileList['barcode'].dropna().unique())

# The processed profiles (downloaded_data/, ~327 MB) are the tier every figure
# actually reads, as opposed to the raw CellProfiler output above. Their paths come
# from utils/data_manifest.tsv — hashed out of downloaded_data/ by
# `python utils/download_data.py --write-manifest` — so this list, the checksums the
# downloader verifies against, and what gets uploaded cannot drift apart.
# Regenerate the manifest first if you have added or removed a table.
import csv as _csv

_manifest = ROOT / 'utils' / 'data_manifest.tsv'
with _manifest.open() as _fh:
    _rows = list(_csv.DictReader([l for l in _fh if not l.startswith('#')],
                                 delimiter='\t'))
assert _rows, f'no rows in {_manifest}; run download_data.py --write-manifest'

processed_profiles = [f"processed_profiles/{r['path']}" for r in sorted(
    _rows, key=lambda r: r['path'])]
print(len(processed_profiles), 'processed profile tables,',
      f"{sum(int(r['size_bytes']) for r in _rows) / 1e6:.0f} MB")

supplementary = [
    'feature_extraction/CP_20230822_100241',
    'feature_extraction/CP_20230823_152219',
    'feature_extraction/HMPSC_FEAT_ICFImg_Cellpose_v2_152219_spheroids_v3.cppipe',
    'image_acquisition/Automatic position detection in 4X 2.ga3',
    'image_acquisition/SelectionOC_20230916.xml',
    'image_acquisition/Spheriod_z_detection_derivative.ga3',
    'image_acquisition/SpheroidDetection.bin',
    'image_acquisition/SpheroidDetection_setup.bin',
    'image_acquisition/filter_settings.tsv',
] + [f'results/{bc}/featICF_{compartment}.parquet'
     for bc in barcodes
     for compartment in ('cells', 'cytoplasm', 'nuclei')] + processed_profiles

FileList = pd.concat(
    [FileList, Detection, pd.DataFrame({'Files': supplementary})],
    ignore_index=True)

# The submitted file writes whole numbers without a trailing ".0"
# (e.g. 3 and 9826528, not 3.0 and 9826528.0); reproduce that here.
def _drop_trailing_zero(v):
    if pd.isna(v):
        return ''
    return str(int(v)) if float(v).is_integer() else str(v)

for col in ['cmpd_conc_um', 'pubchemID']:
    FileList[col] = FileList[col].map(_drop_trailing_zero)

print(len(FileList), 'rows in the combined file list')
FileList.tail()

In [ ]:
# Save the final metadata file.
# The deposited FileList went through a spreadsheet on its way to BIA, which
# left CRLF line endings and wrapped every field containing a comma in quotes
# (compound names such as "Vorinostat (SAHA, MK0683)"). Writing it the same way
# here means a re-run reproduces the deposited file byte-for-byte; plain
# FileList.to_csv(..., sep='\t', index=False) gives the same table otherwise.
def _quote_if_comma(value):
    text = '' if pd.isna(value) else str(value)
    return f'"{text}"' if ',' in text else text

with open('spher_colo52-FileList.tsv', 'w', newline='', encoding='utf-8') as fh:
    fh.write('\t'.join(FileList.columns) + '\r\n')
    for row in FileList.itertuples(index=False):
        fh.write('\t'.join(_quote_if_comma(v) for v in row) + '\r\n')

#### Now prepare the annotations file

In [ ]:
Annotations = pd.read_csv(f'results.tsv', sep='\t')

In [ ]:
Annotations

# # Regex to extract from the filename
Annotations[['barcode','Annotation Type','compartment','well_id', 'z_plane']] = Annotations['Files'].str.extract(r'results\/([A-Z0-9]{8})\/([a-z]{12})\/(.*)_.*_([A-Z0-9]{3})_([0-9]{1,2}).npy')

Annotations.loc[Annotations['compartment'] == 'cell', 'channel'] = 'PHAandWGA'
Annotations.loc[Annotations['compartment'] == 'nuclei', 'channel'] = 'HOECHST'

In [ ]:
# Replace column "Annotation Type" values "segmentation" with "Segmentation masks"
Annotations['Annotation Type'] = Annotations['Annotation Type'].replace(
    {'segmentation': 'Segmentation masks'})

Annotations


In [ ]:
Template.rename(columns={'Files': 'source image'}, inplace=True)

Template

In [ ]:
# Merge the two dataframes on barcode and well_id
Annotations_result = pd.merge(Annotations, Template, on=['barcode', 'well_id', 'z_plane', 'channel'], how='left')

Annotations_result.head()

In [ ]:
# Save the final metadata file
Annotations_result.to_csv(f'spher_colo52-Annotations.tsv', sep= '\t',index=False)